# Step 7 - Dashboard and Visualization

Goal:
- Build an executive dashboard with EDA and ML sections.
- Visualize model comparison, feature importance, confusion matrix, and ROC.
- Export clean files for Power BI and presentation use.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "pyspark": "pyspark",
    "delta": "delta-spark"
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Package installation completed.")
else:
    print("All Step 7 packages are already installed.")

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")
sns.set_palette("tab10")

In [ ]:
APP_NAME = "CreditRisk-Step7-Dashboard"
DEFAULT_WORKSPACE_ROOT = "c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main"

def resolve_workspace_root(default_root: str) -> str:
    candidates = [
        os.getenv("WORKSPACE_ROOT"),
        os.getcwd(),
        default_root,
    ]
    for candidate in candidates:
        if not candidate:
            continue
        candidate = candidate.replace("\\", "/")
        if os.path.isdir(candidate) and os.path.isdir(f"{candidate}/notebooks"):
            return candidate
    return default_root

WORKSPACE_ROOT = resolve_workspace_root(DEFAULT_WORKSPACE_ROOT)
SPARK_LOCAL_DIR = os.getenv("SPARK_LOCAL_DIR", f"{WORKSPACE_ROOT}/tmp/spark-local")
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

def ensure_windows_hadoop_home(workspace_root: str):
    if os.name != "nt":
        return None

    candidate_homes = []
    env_home = os.environ.get("HADOOP_HOME") or os.environ.get("hadoop.home.dir")
    if env_home:
        candidate_homes.append(env_home)
    candidate_homes.append(f"{workspace_root}/third_party/hadoop")

    for home in candidate_homes:
        if not home:
            continue
        winutils_path = os.path.join(home, "bin", "winutils.exe")
        if os.path.exists(winutils_path):
            os.environ["HADOOP_HOME"] = home
            os.environ["hadoop.home.dir"] = home
            os.environ["PATH"] = os.path.join(home, "bin") + ";" + os.environ.get("PATH", "")
            return home

    return None

def reset_pyspark_runtime_state():
    try:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
    except Exception:
        pass

def resolve_delta_base(workspace_root: str) -> str:
    candidates = []
    env_base = os.getenv("DELTA_BASE")
    if env_base:
        candidates.append(env_base)
    candidates.extend([
        f"{workspace_root}/delta_lake",
        "/app/delta_lake",
    ])

    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate.replace("\\", "/")

    return candidates[0].replace("\\", "/")

hadoop_home = ensure_windows_hadoop_home(WORKSPACE_ROOT)
if hadoop_home:
    print(f"Using HADOOP_HOME: {hadoop_home}")

from delta import configure_spark_with_delta_pip

reset_pyspark_runtime_state()
spark_builder = (
    SparkSession.builder
    .master("local[*]")
    .appName(APP_NAME)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", os.getenv("SPARK_LOCAL_SHUFFLE_PARTITIONS", "32"))
    .config("spark.default.parallelism", os.getenv("SPARK_LOCAL_DEFAULT_PARALLELISM", "32"))
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.driver.memory", os.getenv("SPARK_LOCAL_DRIVER_MEMORY", "6g"))
    .config("spark.driver.maxResultSize", os.getenv("SPARK_LOCAL_DRIVER_MAX_RESULT", "1g"))
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

DELTA_BASE = resolve_delta_base(WORKSPACE_ROOT)
FEATURE_PATH = f"{DELTA_BASE}/gold/features"

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Feature path: {FEATURE_PATH}")

In [ ]:
step6_dir = Path(WORKSPACE_ROOT) / "output" / "step6"
step7_dir = Path(WORKSPACE_ROOT) / "output" / "step7"
step7_dir.mkdir(parents=True, exist_ok=True)

required_step6_files = [
    "model_comparison.csv",
    "confusion_matrix_best.csv",
    "roc_curve_best.csv",
    "executive_metrics.json",
    "threshold_tuning_best_model.csv",
    "business_cost_matrix.csv",
]
missing_files = [f for f in required_step6_files if not (step6_dir / f).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing Step 6 outputs: {missing_files}. Run step6_ml_models.ipynb first."
    )

model_comparison_pdf = pd.read_csv(step6_dir / "model_comparison.csv")
cm_pdf = pd.read_csv(step6_dir / "confusion_matrix_best.csv")
roc_pdf = pd.read_csv(step6_dir / "roc_curve_best.csv")
threshold_tuning_pdf = pd.read_csv(step6_dir / "threshold_tuning_best_model.csv")
business_cost_matrix_pdf = pd.read_csv(step6_dir / "business_cost_matrix.csv")

model_metrics_long_pdf = model_comparison_pdf[["model", "accuracy", "f1", "precision", "recall", "auc"]].melt(
    id_vars="model",
    var_name="metric",
    value_name="value",
)

rf_importance_pdf = pd.DataFrame()
gbt_importance_pdf = pd.DataFrame()
if (step6_dir / "feature_importance_rf.csv").exists():
    rf_importance_pdf = pd.read_csv(step6_dir / "feature_importance_rf.csv")
if (step6_dir / "feature_importance_gbt.csv").exists():
    gbt_importance_pdf = pd.read_csv(step6_dir / "feature_importance_gbt.csv")

with open(step6_dir / "executive_metrics.json", "r", encoding="utf-8") as f:
    executive_metrics = json.load(f)

kpi_cards_pdf = pd.DataFrame([
    {"kpi": "Total Loans", "value": executive_metrics.get("total_loans")},
    {"kpi": "Default Rate", "value": executive_metrics.get("default_rate")},
    {"kpi": "Best Model", "value": executive_metrics.get("best_model")},
    {"kpi": "Best AUC", "value": executive_metrics.get("best_auc")},
    {"kpi": "Avg Interest Rate", "value": executive_metrics.get("avg_interest_rate")},
    {"kpi": "Recommended Threshold (Cost)", "value": executive_metrics.get("recommended_threshold_cost")},
    {"kpi": "Recommended Threshold (F1)", "value": executive_metrics.get("recommended_threshold_f1")},
])

feature_parquet_fallback_path = f"{DELTA_BASE}/gold/features_parquet_fallback"
read_errors = []
gold_df = None

try:
    gold_df = spark.read.format("delta").load(FEATURE_PATH)
except Exception as delta_read_error:
    read_errors.append(f"delta:{delta_read_error}")

if gold_df is None:
    parquet_candidates = [FEATURE_PATH, feature_parquet_fallback_path]
    for parquet_path in parquet_candidates:
        if not os.path.exists(parquet_path):
            continue
        try:
            gold_df = spark.read.parquet(parquet_path)
            break
        except Exception as parquet_read_error:
            read_errors.append(f"parquet:{parquet_path}:{parquet_read_error}")

if gold_df is None:
    raise RuntimeError(
        "Step 7 could not load feature table from delta/parquet. "
        f"Errors: {read_errors}"
    )

if "target" in gold_df.columns and "label" not in gold_df.columns:
    gold_df = gold_df.withColumn("label", F.col("target").cast("int"))
elif "label" in gold_df.columns:
    gold_df = gold_df.withColumn("label", F.col("label").cast("int"))
else:
    raise ValueError("Gold feature table must include target or label column.")

print(f"Step 6 artifacts loaded from: {step6_dir.as_posix()}")
print(f"Gold rows available for Step 7: {gold_df.count():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor("#101820")
ax.set_facecolor("#101820")
ax.axis("off")

avg_interest_raw = executive_metrics.get("avg_interest_rate")
avg_interest_display = f"{float(avg_interest_raw):.2f}%" if avg_interest_raw is not None else "N/A"

cards = [
    ("Total Loans", f"{int(executive_metrics.get('total_loans', 0)):,}"),
    ("Default Rate", f"{100 * float(executive_metrics.get('default_rate', 0.0)):.2f}%"),
    ("Best Model", str(executive_metrics.get('best_model', 'N/A'))),
    ("Best AUC", f"{float(executive_metrics.get('best_auc', 0.0)):.4f}"),
    ("Avg Interest Rate", avg_interest_display),
]

card_width = 0.18
card_height = 0.55
start_x = 0.02
gap = 0.015

for i, (title, value) in enumerate(cards):
    x = start_x + i * (card_width + gap)
    rect = plt.Rectangle((x, 0.2), card_width, card_height, color="#1f3b4d", alpha=0.95, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x + 0.02, 0.62, title, fontsize=13, color="#8fd3ff", transform=ax.transAxes, weight="bold")
    ax.text(x + 0.02, 0.42, value, fontsize=18, color="#ffffff", transform=ax.transAxes, weight="bold")

insight_text = (
    "Business Insight: Higher interest burden and weaker credit profile segments align with higher default risk. "
    "Use best model score as early-warning signal in loan approval flow."
)
ax.text(0.02, 0.05, insight_text, fontsize=12, color="#d6eaff", transform=ax.transAxes)

plt.title("Executive Overview", color="#ffffff", fontsize=20, loc="left")
plt.tight_layout()
plt.savefig(step7_dir / "page1_executive_overview.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
income_pdf = pd.DataFrame(columns=["annual_inc"])
if "annual_inc" in gold_df.columns:
    income_pdf = (
        gold_df
        .select(F.col("annual_inc").cast("double").alias("annual_inc"))
        .dropna()
        .sample(False, 0.20, seed=42)
        .limit(250000)
        .toPandas()
    )

if not income_pdf.empty:
    p99_income = income_pdf["annual_inc"].quantile(0.99)
    income_pdf["annual_inc_clip"] = income_pdf["annual_inc"].clip(upper=p99_income)

status_pdf = gold_df.groupBy("label").count().orderBy("label").toPandas()
status_pdf["segment"] = status_pdf["label"].map({0: "Non-Default", 1: "Default"})

if "issue_date" in gold_df.columns:
    issue_col = F.col("issue_date").cast("string")
    issue_date_parsed = F.coalesce(
        F.to_date(issue_col),
        F.to_date(issue_col, "yyyy-MM-dd"),
        F.to_date(issue_col, "MMM-yyyy"),
        F.to_date(issue_col, "MM/yyyy"),
    )

    monthly_default_pdf = (
        gold_df
        .withColumn("issue_month", F.date_format(issue_date_parsed, "yyyy-MM"))
        .where(F.col("issue_month").isNotNull())
        .groupBy("issue_month")
        .agg(
            F.avg(F.col("label")).alias("default_rate"),
            F.count("*").alias("loan_count"),
        )
        .orderBy("issue_month")
        .toPandas()
    )
else:
    monthly_default_pdf = pd.DataFrame(columns=["issue_month", "default_rate", "loan_count"])

if "fico_range_high" in gold_df.columns:
    fico_risk_pdf = (
        gold_df
        .where(F.col("fico_range_high").isNotNull())
        .withColumn("fico_band", (F.floor(F.col("fico_range_high") / 20) * 20).cast("int"))
        .groupBy("fico_band")
        .agg(F.avg("label").alias("default_rate"), F.count("*").alias("loan_count"))
        .orderBy("fico_band")
        .toPandas()
    )
else:
    fico_risk_pdf = pd.DataFrame(columns=["fico_band", "default_rate", "loan_count"])

if "home_ownership_idx" in gold_df.columns:
    home_risk_pdf = (
        gold_df
        .where(F.col("home_ownership_idx").isNotNull())
        .withColumn("home_ownership_idx", F.col("home_ownership_idx").cast("int"))
        .groupBy("home_ownership_idx")
        .agg(F.avg("label").alias("default_rate"), F.count("*").alias("loan_count"))
        .orderBy("home_ownership_idx")
        .toPandas()
    )
else:
    home_risk_pdf = pd.DataFrame(columns=["home_ownership_idx", "default_rate", "loan_count"])

if "purpose_idx" in gold_df.columns:
    purpose_risk_pdf = (
        gold_df
        .where(F.col("purpose_idx").isNotNull())
        .withColumn("purpose_idx", F.col("purpose_idx").cast("int"))
        .groupBy("purpose_idx")
        .agg(F.avg("label").alias("default_rate"), F.count("*").alias("loan_count"))
        .orderBy("purpose_idx")
        .toPandas()
    )
else:
    purpose_risk_pdf = pd.DataFrame(columns=["purpose_idx", "default_rate", "loan_count"])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.patch.set_facecolor("#0f172a")

for ax in axes.flatten():
    ax.set_facecolor("#eaf4ff")

ax = axes[0, 0]
if not income_pdf.empty:
    sns.histplot(income_pdf["annual_inc_clip"], bins=40, kde=False, color="#0d6efd", ax=ax)
    ax.set_title("Income Distribution (clipped at P99)")
    ax.set_xlabel("Annual Income")
    ax.set_ylabel("Count")
else:
    ax.text(0.5, 0.5, "annual_inc not available", ha="center", va="center")

ax = axes[0, 1]
if not status_pdf.empty:
    ax.pie(status_pdf["count"], labels=status_pdf["segment"], autopct="%1.1f%%", colors=["#4caf50", "#ff7043"])
    ax.set_title("Loan Status Distribution")
else:
    ax.text(0.5, 0.5, "label distribution unavailable", ha="center", va="center")

ax = axes[0, 2]
if not monthly_default_pdf.empty:
    monthly_plot = monthly_default_pdf.tail(48)
    ax.plot(monthly_plot["issue_month"], monthly_plot["default_rate"], color="#0d47a1", linewidth=2)
    ax.set_title("Monthly Default Trend")
    ax.set_xlabel("Issue Month")
    ax.set_ylabel("Default Rate")
    ax.tick_params(axis="x", rotation=60)
else:
    ax.text(0.5, 0.5, "issue_date trend unavailable", ha="center", va="center")

ax = axes[1, 0]
if not fico_risk_pdf.empty:
    sns.lineplot(data=fico_risk_pdf, x="fico_band", y="default_rate", marker="o", color="#1565c0", ax=ax)
    ax.set_title("FICO Band vs Default Risk")
    ax.set_xlabel("FICO Band")
    ax.set_ylabel("Default Rate")
else:
    ax.text(0.5, 0.5, "fico_range_high unavailable", ha="center", va="center")

ax = axes[1, 1]
if not home_risk_pdf.empty:
    sns.barplot(data=home_risk_pdf, x="home_ownership_idx", y="default_rate", color="#1976d2", ax=ax)
    ax.set_title("Home Ownership Segment Risk")
    ax.set_xlabel("home_ownership_idx")
    ax.set_ylabel("Default Rate")
else:
    ax.text(0.5, 0.5, "home_ownership_idx unavailable", ha="center", va="center")

ax = axes[1, 2]
if not purpose_risk_pdf.empty:
    top_purpose = purpose_risk_pdf.sort_values("loan_count", ascending=False).head(12)
    sns.barplot(data=top_purpose, x="purpose_idx", y="default_rate", color="#1e88e5", ax=ax)
    ax.set_title("Loan Purpose Segment Risk")
    ax.set_xlabel("purpose_idx")
    ax.set_ylabel("Default Rate")
else:
    ax.text(0.5, 0.5, "purpose_idx unavailable", ha="center", va="center")

plt.suptitle("EDA Dashboard (Step 7)", color="#ffffff", fontsize=20)
plt.tight_layout()
plt.savefig(step7_dir / "page2_eda_dashboard.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.patch.set_facecolor("#0f172a")
for ax in axes.flatten():
    ax.set_facecolor("#f5fbff")

ax = axes[0, 0]
core_metrics_pdf = model_metrics_long_pdf[model_metrics_long_pdf["metric"].isin(["accuracy", "f1", "auc"])]
sns.barplot(data=core_metrics_pdf, x="model", y="value", hue="metric", ax=ax)
ax.set_title("Model Comparison (Accuracy / F1 / AUC)")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=20)

ax = axes[0, 1]
if not rf_importance_pdf.empty:
    top_importance = rf_importance_pdf.head(10).sort_values("importance", ascending=True)
    title = "Top 10 Feature Importance (RandomForest)"
elif not gbt_importance_pdf.empty:
    top_importance = gbt_importance_pdf.head(10).sort_values("importance", ascending=True)
    title = "Top 10 Feature Importance (GBT)"
else:
    top_importance = pd.DataFrame(columns=["feature", "importance"])
    title = "Feature Importance Not Available"

if not top_importance.empty:
    sns.barplot(data=top_importance, x="importance", y="feature", orient="h", color="#1976d2", ax=ax)
    ax.set_title(title)
else:
    ax.text(0.5, 0.5, "No feature importance file", ha="center", va="center")
    ax.set_title(title)

ax = axes[1, 0]
cm_matrix = (
    cm_pdf
    .pivot(index="label", columns="prediction", values="count")
    .reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    .fillna(0)
    .astype(int)
)
sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Confusion Matrix (Best Model)")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

ax = axes[1, 1]
ax.plot(roc_pdf["fpr"], roc_pdf["tpr"], color="#1565c0", linewidth=2, label=executive_metrics.get("best_model", "Best Model"))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_title("ROC Curve (Best Model)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()

plt.suptitle("ML Dashboard (Step 7)", color="#ffffff", fontsize=20)
plt.tight_layout()
plt.savefig(step7_dir / "page3_ml_dashboard.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor("#0f172a")
for ax in axes.flatten():
    ax.set_facecolor("#f5fbff")

ax = axes[0]
ax.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["f1"], marker="o", label="F1")
ax.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["recall"], marker="o", label="Recall")
ax.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["precision"], marker="o", label="Precision")
ax2 = ax.twinx()
ax2.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["expected_cost"], color="#ef6c00", linewidth=2, label="Expected Cost")

best_cost_threshold = float(executive_metrics.get("recommended_threshold_cost", threshold_tuning_pdf["threshold"].iloc[0]))
ax.axvline(best_cost_threshold, linestyle="--", color="red", alpha=0.8)
ax.set_title("Threshold Strategy (Metrics + Cost)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metric Value")
ax2.set_ylabel("Expected Cost")

left_handles, left_labels = ax.get_legend_handles_labels()
right_handles, right_labels = ax2.get_legend_handles_labels()
ax.legend(left_handles + right_handles, left_labels + right_labels, loc="lower center", ncol=2)

ax = axes[1]
scenario_heatmap = business_cost_matrix_pdf.pivot(index="fn_cost", columns="fp_cost", values="optimal_threshold")
sns.heatmap(scenario_heatmap, annot=True, fmt=".2f", cmap="Blues", cbar=True, ax=ax)
ax.set_title("Optimal Threshold by Cost Scenario")
ax.set_xlabel("FP Cost")
ax.set_ylabel("FN Cost")

plt.suptitle("Threshold Tuning and Business Cost Matrix", color="#ffffff", fontsize=18)
plt.tight_layout()
plt.savefig(step7_dir / "page4_threshold_strategy.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
dashboard_pack_dir = step7_dir / "dashboard_pack"
dashboard_pack_dir.mkdir(parents=True, exist_ok=True)

model_comparison_pdf.to_csv(dashboard_pack_dir / "model_comparison.csv", index=False)
model_metrics_long_pdf.to_csv(dashboard_pack_dir / "model_metrics_long.csv", index=False)
cm_pdf.to_csv(dashboard_pack_dir / "confusion_matrix_best.csv", index=False)
roc_pdf.to_csv(dashboard_pack_dir / "roc_curve_best.csv", index=False)
threshold_tuning_pdf.to_csv(dashboard_pack_dir / "threshold_tuning_best_model.csv", index=False)
business_cost_matrix_pdf.to_csv(dashboard_pack_dir / "business_cost_matrix.csv", index=False)
kpi_cards_pdf.to_csv(dashboard_pack_dir / "kpi_cards.csv", index=False)

if not rf_importance_pdf.empty:
    rf_importance_pdf.to_csv(dashboard_pack_dir / "feature_importance_rf.csv", index=False)
if not gbt_importance_pdf.empty:
    gbt_importance_pdf.to_csv(dashboard_pack_dir / "feature_importance_gbt.csv", index=False)

status_pdf.to_csv(dashboard_pack_dir / "loan_status_distribution.csv", index=False)
monthly_default_pdf.to_csv(dashboard_pack_dir / "monthly_default_trend.csv", index=False)
fico_risk_pdf.to_csv(dashboard_pack_dir / "fico_default_risk.csv", index=False)
home_risk_pdf.to_csv(dashboard_pack_dir / "home_ownership_risk.csv", index=False)
purpose_risk_pdf.to_csv(dashboard_pack_dir / "loan_purpose_risk.csv", index=False)

visual_manifest_pdf = pd.DataFrame([
    {"page": "Executive Overview", "visual": "KPI Cards", "dataset": "kpi_cards.csv"},
    {"page": "Executive Overview", "visual": "Loan Status Pie", "dataset": "loan_status_distribution.csv"},
    {"page": "EDA Dashboard", "visual": "Income Histogram", "dataset": "gold feature sample in notebook"},
    {"page": "EDA Dashboard", "visual": "Monthly Default Trend", "dataset": "monthly_default_trend.csv"},
    {"page": "EDA Dashboard", "visual": "FICO Risk Line", "dataset": "fico_default_risk.csv"},
    {"page": "EDA Dashboard", "visual": "Home Ownership Risk", "dataset": "home_ownership_risk.csv"},
    {"page": "EDA Dashboard", "visual": "Loan Purpose Risk", "dataset": "loan_purpose_risk.csv"},
    {"page": "ML Dashboard", "visual": "Model Metrics Grouped Bar", "dataset": "model_metrics_long.csv"},
    {"page": "ML Dashboard", "visual": "Feature Importance", "dataset": "feature_importance_rf.csv or feature_importance_gbt.csv"},
    {"page": "ML Dashboard", "visual": "Confusion Matrix", "dataset": "confusion_matrix_best.csv"},
    {"page": "ML Dashboard", "visual": "ROC Curve", "dataset": "roc_curve_best.csv"},
    {"page": "Threshold Strategy", "visual": "Threshold vs Metrics", "dataset": "threshold_tuning_best_model.csv"},
    {"page": "Threshold Strategy", "visual": "Business Cost Matrix", "dataset": "business_cost_matrix.csv"},
])
visual_manifest_pdf.to_csv(dashboard_pack_dir / "visual_manifest.csv", index=False)

with open(dashboard_pack_dir / "executive_metrics.json", "w", encoding="utf-8") as f:
    json.dump(executive_metrics, f, indent=2)

print(f"Dashboard pack exported to: {dashboard_pack_dir.as_posix()}")
print("Power BI pages recommended:")
print("1) Executive Overview -> kpi_cards.csv + loan_status_distribution.csv")
print("2) EDA Dashboard -> monthly_default_trend.csv + fico_default_risk.csv + home_ownership_risk.csv + loan_purpose_risk.csv")
print("3) ML Dashboard -> model_metrics_long.csv + feature_importance_*.csv + confusion_matrix_best.csv + roc_curve_best.csv")
print("4) Threshold Strategy -> threshold_tuning_best_model.csv + business_cost_matrix.csv")

## Step 7 Summary

This notebook produces:
- Executive KPI page
- EDA dashboard page with required chart types
- ML dashboard page with grouped model comparison, feature importance, confusion matrix, and ROC
- Threshold strategy page for decision policy
- Optimized Power BI export package with visual manifest